This notebook annotates and subclusters neural stem cell-like cells based on two marker gene panels.

Input: .h5ad with annotated major cell type lineages in ../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_global_annotations.h5ad  
Output: .h5ad with fine-grained neural stem cell annotations in ../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad  

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
sc.settings.set_figure_params(dpi=300)

In [ ]:
adata = sc.read_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_global_annotations.h5ad')
adata

In [ ]:
import matplotlib.pyplot as plt

# Define markers
dumitru_pos = ['NES', 'ASCL1', 'SOX2', 'PAX6']
dumitru_neg = ['S100B', 'SOX10', 'OLIG2']
esser_pos = ['SOX2', 'HOPX', 'ID4', 'GLI3', 'LRRC3B', 'RHOJ', 'SLC4A4']
esser_neg = ['S100B', 'SOX10', 'OLIG2', 'EGFR']

# Assign NSC types
adata.obs['dumitru_nsc'] = ((adata[:, dumitru_pos].layers['counts'] > 0).sum(axis=1) == len(dumitru_pos)) & \
                            ((adata[:, dumitru_neg].layers['counts'] > 0).sum(axis=1) == 0)
adata.obs['new_nsc'] = ((adata[:, esser_pos].layers['counts'] > 0).sum(axis=1) == len(esser_pos)) & \
                       ((adata[:, esser_neg].layers['counts'] > 0).sum(axis=1) == 0)

# Create category
adata.obs['nsc_type'] = 'Other'
adata.obs.loc[adata.obs['dumitru_nsc'], 'nsc_type'] = 'Dumitru NSC'
adata.obs.loc[adata.obs['new_nsc'], 'nsc_type'] = 'Esser NSC'

# Sort and plot
category_order = {'Other': 0, 'Dumitru NSC': 1, 'Esser NSC': 2}
plot_order = adata.obs['nsc_type'].map(category_order).sort_values().index

palette = {'Other': 'lightgray', 'Dumitru NSC': 'purple', 'Esser NSC': '#00DCFF'}
size_map = {'Other': 1, 'Dumitru NSC': 30, 'Esser NSC': 30}

sc.pl.umap(adata[plot_order], color='nsc_type', palette=palette, 
           size=adata[plot_order].obs['nsc_type'].map(size_map), 
           title='', frameon=False)

In [ ]:
adata.obs['dumitru_nsc'].value_counts()

In [ ]:
adata.obs['new_nsc'].value_counts()

In [ ]:
nsc_adata = adata[(adata.obs['new_nsc'] & adata.obs['leiden'].isin(['1'])) | (adata.obs['dumitru_nsc'] & adata.obs['leiden'].isin(['1']))].copy()
nsc_adata

In [ ]:
import scanpy.external as sce

# Harmony batch correction (creates X_pca_harmony)
sc.tl.pca(nsc_adata)
sce.pp.harmony_integrate(nsc_adata, key="patient_id", max_iter_harmony=40)

# Neighbors / UMAP / clustering on Harmony PCs
sc.pp.neighbors(nsc_adata, use_rep="X_pca_harmony")
sc.tl.umap(nsc_adata, min_dist=0.85)

In [ ]:
sc.pl.umap(nsc_adata, color=['donor'])
sc.pl.umap(nsc_adata, color=['age'])
sc.pl.umap(nsc_adata, color=['total_counts'])
sc.pl.umap(nsc_adata, color=['n_genes_by_counts'])
sc.pl.umap(nsc_adata, color=['pct_counts_mt'])
sc.pl.umap(nsc_adata, color=['dumitru_nsc'])

In [ ]:
# qNSC markers
sc.pl.umap(nsc_adata, color=['ETNPPL', 'PDGFRB', 'RHOJ', 'HOPX', 
                             'HES1', 'HES5', 'MFGE8', 'ID4', 
                             'SLC4A4', 'LRRC3B', 'GRM3', 'CLU'], ncols=4)

In [ ]:
# aNSC markers
sc.pl.umap(nsc_adata, color=['SOX2', 'PAX6', 'TNC', 'VIM', 
                             'NES', 'PROX1', 'ASCL1', 'EGFR', 
                             'PTPRD', 'STMN1', 'EOMES', 'MKI67'], ncols=4)

In [ ]:
sc.tl.leiden(nsc_adata, resolution=0.1)
sc.pl.umap(nsc_adata, color=['leiden'], legend_loc='on data')

In [ ]:
leiden_mapping = {
    '0': 'nsc_01',
    '1': 'nsc_03',
    '2': 'nsc_02',
    '3': 'nsc_04',
    '4': 'nsc_05'
}

nsc_adata.obs['nsc'] = nsc_adata.obs['leiden'].map(leiden_mapping)
nsc_adata.uns['nsc_colors'] = ['darkblue', 'green', 'blue', 'orange', 'red']

sc.pl.umap(nsc_adata, color=['nsc'], title='')

In [ ]:
sc.tl.rank_genes_groups(nsc_adata, groupby='nsc')
sc.pl.rank_genes_groups_dotplot(nsc_adata, n_genes=10)

In [ ]:
# Get indices of NSC cells
nsc_indices = nsc_adata.obs_names

# ensure string dtype for the target column
adata.obs['cell_type'] = adata.obs['cell_type'].astype(str)

# assign only where subset is not null; others keep original cell_type
mask = nsc_adata.obs['nsc'].notna()
adata.obs.loc[nsc_indices[mask], 'cell_type'] = nsc_adata.obs.loc[mask, 'nsc'].astype(str).values

print(adata.obs['cell_type'].value_counts().sort_index())

In [ ]:
adata.write_h5ad('../data/dumitru/single_nucleus/h5ad/dumitru_all_donors_nsc_annotations.h5ad')